|<h2>Course:</h2>|<h1><a href="https://derivingsystems.com/course.html" target="_blank">Build your own vLLM: inference engines from the memory system up</a></h1>|
|-|:-:|
|<h2>Part 6:</h2>|<h1>The Server<h1>|
|<h2>Section:</h2>|<h1>Observability<h1>|
|<h2>Lecture:</h2>|<h1><b>CodeChallenge: six numbers, and what each one is asking you to fix<b></h1>|

<br>

<h5><b>Course repo:</b> <a href="https://github.com/Venugopalan2610/vllm-from-scratch" target="_blank">github.com/Venugopalan2610/vllm-from-scratch</a></h5>
<h5><b>The derivations:</b> <a href="https://derivingsystems.com" target="_blank">derivingsystems.com</a></h5>
<i>The notebooks build the intuition. The ladder in app/ makes you build the thing.</i>

In [ ]:
import numpy as np
import matplotlib.pyplot as plt

import matplotlib_inline.backend_inline
matplotlib_inline.backend_inline.set_matplotlib_formats('svg')

rng = np.random.default_rng(0)

Compute the six numbers from a recorded run, then use them to answer a
question a single latency number cannot.

All arithmetic on arrays. The interesting part is Exercise 3.

In [ ]:
### run this cell: one server run, already recorded

n = 4000
arrive   = np.cumsum(rng.exponential(1/25, size=n))
queue    = rng.exponential(0.8, size=n)                 # wait before first step
out_len  = rng.lognormal(mean=np.log(120), sigma=0.9, size=n).astype(int) + 1
itl      = rng.lognormal(mean=np.log(0.012), sigma=0.4, size=n)  # per-token, s
prefill  = 0.04 * rng.lognormal(mean=np.log(400), sigma=0.6, size=n)/1000

first_token = arrive + queue + prefill
finish      = first_token + out_len*itl
print(f'{n} requests over {finish.max():.0f} seconds')

# Exercise 1: the metrics

TTFT, TPOT, end-to-end latency, throughput, queue wait. Percentiles, not
means.

In [ ]:
ttft    = first_token - arrive
tpot    = itl
latency = finish - arrive
tokens  = out_len.sum()
wall    = finish.max()

def q(x, p): return float(np.percentile(x, p))

print(f"{'metric':<12} {'p50':>9} {'p90':>9} {'p99':>9}")
for name, x in (('TTFT (s)', ttft), ('TPOT (ms)', tpot*1000),
                ('latency (s)', latency)):
  print(f'{name:<12} {q(x,50):>9.3f} {q(x,90):>9.3f} {q(x,99):>9.3f}')

print(f'\nthroughput  {tokens/wall:8.0f} tokens/s, {n/wall:.1f} requests/s')
print(f'queue wait  {np.median(queue):8.3f} s median')

# Exercise 2: goodput against three promises

A request is good only if it met **both** its TTFT and TPOT targets. Count
those, and compare with raw throughput.

In [ ]:
def goodput(ttft_sla, tpot_sla):
  ok = (ttft <= ttft_sla) & (tpot <= tpot_sla)
  return ok.mean(), ok.sum()/wall

print(f"{'TTFT SLA':>9} {'TPOT SLA':>10} {'met':>7} {'goodput req/s':>14}")
for ts, ps in ((1.0, 0.020), (2.0, 0.030), (5.0, 0.050)):
  frac, gp = goodput(ts, ps)
  print(f'{ts:>8.1f}s {ps*1000:>9.0f}ms {100*frac:>6.1f}% {gp:>14.1f}')
print(f'\nraw throughput {n/wall:.1f} requests/s, and it does not move')

# Exercise 3: which promise did they break?

This is the exercise. A request can miss because it waited to start, or
because it generated slowly, and those are different bugs with different
fixes.

In [ ]:
# who is failing? split the misses by which promise they broke
sla_t, sla_p = 1.0, 0.020
slow_start = (ttft > sla_t) & (tpot <= sla_p)
slow_tokens = (ttft <= sla_t) & (tpot > sla_p)
both       = (ttft > sla_t) & (tpot > sla_p)

print(f'missed on TTFT only:  {100*slow_start.mean():5.1f}%  -> queueing or prefill')
print(f'missed on TPOT only:  {100*slow_tokens.mean():5.1f}%  -> the step is too slow')
print(f'missed on both:       {100*both.mean():5.1f}%')

print(f'\nof the TTFT misses, queue wait is {100*np.median(queue[ttft>sla_t])/np.median(ttft[ttft>sla_t]):.0f}% '
      f'of the median TTFT')

# Exercise 4: look at the shapes

In [ ]:
fig, axs = plt.subplots(1, 3, figsize=(15,4))
axs[0].hist(ttft, bins=60); axs[0].set(xlabel='TTFT (s)', ylabel='requests',
                                       title='Time to first token')
axs[0].axvline(sla_t, color='r', ls='--')
axs[1].hist(tpot*1000, bins=60); axs[1].set(xlabel='TPOT (ms)', title='Per output token')
axs[1].axvline(sla_p*1000, color='r', ls='--')
axs[2].hist(latency, bins=60); axs[2].set(xlabel='Total latency (s)', title='End to end')
for a in axs: a.grid(alpha=.3)
plt.tight_layout(); plt.show()

print(f'TTFT    mean {ttft.mean():.3f}s  median {np.median(ttft):.3f}s  '
      f'p99 {np.percentile(ttft,99):.3f}s')
print(f'the mean is {ttft.mean()/np.median(ttft):.1f}x the median: a long right tail')

### What you can now answer that a latency number cannot

Exercise 3 is the one that earns its place. "p99 latency is 12 seconds" is
a fact with no action attached to it. Splitting the misses gives you one:

- **mostly TTFT misses** means requests are waiting, so the fix is
  admission and concurrency, and probably prefix caching for the prefill
  half. Stages 09, 10 and 11.
- **mostly TPOT misses** means the step itself is slow, so the fix is the
  kernel, the batch size, or CUDA graphs. Stages 08 and 12.
- **both** means you are past capacity and no amount of tuning helps.
  Add hardware or shed load.

Three completely different projects, and the single number you started
with pointed at none of them.

### And the mean is a trap

Look at the ratio in Exercise 4. The mean TTFT is well above the median,
because the distribution has a long right tail, which every queueing
system does. A dashboard that reports averages will look healthy through
an outage that half your users are experiencing.

Report percentiles. And know which population they are over, which is the
trap Part 4's chunked-prefill challenge set for you.

    ./vc guide 16